<a href="https://colab.research.google.com/github/ese-ada-lovelace-2025/irp-wm722/blob/main/DEMDownscaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import rasterio
import numpy as np
from google.colab import drive
import sys
import glob
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union
import geopandas as gpd

drive.mount('/content/drive/')
sys.path.append('/content/drive/MyDrive/IRP/scripts')

Mounted at /content/drive/


In [ ]:
DEM_path1 = '/content/drive/MyDrive/IRP/Test/UK/DEM.tif'
DEM_path2 = '/content/drive/MyDrive/IRP/Test/UK/DEMWest.tif'
DEM_path3 = '/content/drive/MyDrive/IRP/Test/UK/DEMEast.tif'
out_path = '/content/drive/MyDrive/IRP/Test/UK/DEMCompressed.tif'

In [ ]:
from rasterio.merge import merge
from rasterio.enums import Resampling

paths = [
    DEM_path1,
    DEM_path2,
    DEM_path3,
]

# open DEMs
srcs = [rasterio.open(p) for p in paths]

# merge and resample
mosaic, transform = merge(
    srcs,
    res=10,
    resampling=Resampling.average,
    target_aligned_pixels=True,
)

# copy metadata from first DEM
profile = srcs[0].profile.copy()

profile.update(
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    transform=transform,
    dtype=mosaic.dtype,
)

# write combined DEM
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(mosaic)

for src in srcs:
    src.close()

In [ ]:
from google.colab import runtime
runtime.unassign()

# Calculating AOI for WoE and MC analysis

In [2]:
DEM = '/content/drive/MyDrive/IRP/Test/UK/QDEM.tif'

# AI assistace with rasterio functionality for getting EXACT area polygon, not bounding box
# bbox will make any spatial analysis for random distribution vs observed to be useless.
with rasterio.open(DEM) as src:

    # 255 = valid data and 0 = invalid/nodata
    mask = src.dataset_mask()

    geoms = []

    for geom, value in shapes(
        mask,
        mask=(mask > 0),
        transform=src.transform
    ):
        if value > 0:
            geoms.append(shape(geom))

    # Merge all valid pieces into one polygon
    study_area = unary_union(geoms)

    raster_crs = src.crs

print(study_area) # as tuple
print("Area:", study_area.area)

POLYGON ((300000 155000, 300000 120000, 340000 120000, 340000 155000, 300000 155000))
Area: 1400000000.0
